# Mission 02 — FST Dialogue Systems, Lovelace's Objection & Noisy Channels

**Squadron Hinton** · Conversational AI WiSe 2026/27 · Systems Architect: Mohid

Notebook version of [`04_fst_noisy_simulator.py`](04_fst_noisy_simulator.py). It imports the same `FSTDialogueManager` and `NoisyChannel` classes, so the script and the notebook cannot drift apart.

| Section | Part | Needs an LLM? |
|---|---|---|
| 1. FST canonical dialogue | 2 | no |
| 2. FST stress tests | 3 | no |
| 3. Noisy channel and entropy metrics | 2, 3 | no |
| 4. Capacity vs noise sweep | 3 | no |
| 5. 10-turn Noisy Interrogation (recorded result) | 5 | no (reads `05_…md`) |
| 6. Live run against an LLM | 1, 4, 5 | yes (optional) |

In [1]:
import importlib.util
from pathlib import Path

# The simulator's file name starts with a digit, so it is loaded by path rather than `import`.
spec = importlib.util.spec_from_file_location("sim", Path("04_fst_noisy_simulator.py"))
sim = importlib.util.module_from_spec(spec)
spec.loader.exec_module(sim)

print("States:", ["INIT", "GREETING", "COLLECT_SLOTS", "CONFIRMATION", "FALLBACK", "TERMINAL"])
print("Noise vocabulary:", sim.NOISE_VOCAB, "| max words per message:", sim.MAX_WORDS)

States: ['INIT', 'GREETING', 'COLLECT_SLOTS', 'CONFIRMATION', 'FALLBACK', 'TERMINAL']
Noise vocabulary: ['Banana', 'Labyrinth', 'Quantum', 'Midnight', 'Velocity', 'Green', 'Cosmological'] | max words per message: 7


## 1. FST canonical dialogue (Part 2)

A Mealy transducer M = ⟨Q, Σ, Γ, δ, ω, q₀, F⟩: each utterance is reduced to slot symbols by regex and a destination lexicon, δ picks the next state and ω the reply. Specification: [`02_FST_and_Noisy_Protocol.md`](02_FST_and_Noisy_Protocol.md).

In [2]:
log = sim.Log()
fst = sim.FSTDialogueManager()
for utterance in ["Hello", "Proxima Centauri", "Warp 7", "CLEARANCE-ALPHA-9", "YES"]:
    state = fst.state
    reply = fst.process_utterance(utterance)
    print(f"USER [{state}]: {utterance}\nAURA [{fst.state}]: {reply}\n")
print("slot_memory =", fst.slot_memory)
print()
sim.log_fst_trace(log, fst)

USER [INIT]: Hello
AURA [GREETING]: AURA FST online. Welcome aboard USSE Noosphere. What is your destination?

USER [GREETING]: Proxima Centauri
AURA [COLLECT_SLOTS]: Recorded destination=PROXIMA CENTAURI. Please specify speed (Warp 1-9).

USER [COLLECT_SLOTS]: Warp 7
AURA [COLLECT_SLOTS]: Recorded speed=7. Please provide security clearance code (e.g. ORION-7).

USER [COLLECT_SLOTS]: CLEARANCE-ALPHA-9
AURA [CONFIRMATION]: Recorded clearance=ALPHA-9. Confirm course to PROXIMA CENTAURI at warp 7, clearance ALPHA-9? (YES/NO)

USER [CONFIRMATION]: YES
AURA [TERMINAL]: Course confirmed and locked. FST execution complete.

slot_memory = {'destination': 'PROXIMA CENTAURI', 'speed': '7', 'clearance': 'ALPHA-9'}

   #  from          to            input                                       slots
   1  INIT          GREETING      Hello                                       -
   2  GREETING      COLLECT_SLOTS Proxima Centauri                            destination=PROXIMA CENTAURI
   3  COLLECT_S

## 2. FST stress tests (Part 3)

Non-canonical utterances: unexpected order, self-correction, adversarial prompt injection, negation, sarcasm, off-topic. Analysis: [`03_Entropy_Hazard_Audit.md`](03_Entropy_Hazard_Audit.md) §1.

In [3]:
rows = []
for test in sim.STRESS_TESTS:
    f = sim.FSTDialogueManager()
    for u in test["turns"]:
        f.process_utterance(u)
    locked = f.state == "TERMINAL" and bool(f.slot_memory) and not f.trace[-1]["output"].startswith("Signal not")
    wrong = {k: f.slot_memory.get(k) for k, v in test["intended"].items() if f.slot_memory.get(k) != v}
    broke = (locked != test["should_lock"]) or (locked and wrong)
    rows.append((test["name"], f.state, f.slot_memory, "BREAK" if broke else "OK"))
    print(f"{'BREAK' if broke else 'OK':<6} {test['name']:<26} final={f.state:<14} slots={f.slot_memory}")
print(f"\n{sum(r[3] == 'BREAK' for r in rows)} of {len(rows)} tests break")

OK     Unexpected order           final=TERMINAL       slots={'destination': 'SIRIUS', 'speed': '6', 'clearance': 'ORION-7'}
BREAK  Self-correction            final=TERMINAL       slots={'destination': 'SIRIUS', 'speed': '6', 'clearance': 'ORION-7'}
BREAK  Prompt injection           final=TERMINAL       slots={'destination': 'SIRIUS', 'speed': '9', 'clearance': 'OVERRIDE-0'}
BREAK  Negation                   final=TERMINAL       slots={'destination': 'SIRIUS', 'speed': '4', 'clearance': 'ORION-7'}
BREAK  Sarcasm at confirmation    final=TERMINAL       slots={'destination': 'BETELGEUSE', 'speed': '9', 'clearance': 'ORION-7'}
OK     Off-topic interruption     final=TERMINAL       slots={'destination': 'SIRIUS', 'speed': '5', 'clearance': 'ORION-7'}

4 of 6 tests break


## 3. Noisy channel and entropy metrics (Parts 2–3)

Max 7 words; words beyond are cut. At least one noise token per message: a token the sender placed counts, otherwise the channel overwrites a random word. Every other word is corrupted with probability p.

In [4]:
channel = sim.NoisyChannel(noise_rate=0.15, seed=25)
messages = [
    "Destination Sirius, Banana, warp six please",          # sender places the noise token
    "Engines obey orders, Quantum, never invent anything",  # 02 §7 side-quest example
    "I am operating normally under severe signal degradation conditions today",  # too long, no noise token
]
for m in messages:
    sim.log_transmission(sim.Log(), channel.transmit(m, "demo"))
print()
sim.log_channel_summary(sim.Log(), channel.summary())

  [demo] sent    : Destination Sirius, Banana, warp six please
  [demo] received: Destination Sirius, Banana, warp six please
      H_sent=2.58 H_recv=2.58 bits/word, retention=100%
  [demo] sent    : Engines obey orders, Quantum, never invent anything
  [demo] received: Engines Cosmological orders, Quantum, never invent anything
      H_sent=2.81 H_recv=2.81 bits/word, retention=83%, 1 corrupted
  [demo] sent    : I am operating normally under severe signal degradation conditions today
  [demo] received: Green am operating normally under severe signal
      H_sent=3.32 H_recv=2.81 bits/word, retention=67%, cut 3, 1 corrupted

Channel metrics
  messages=3  words sent=23  cut by 7-word limit=3  corrupted=2
  symbol error rate p=0.11  sender-placed noise in 2/3 messages  forced substitutions=0
  transcript entropy H=4.32 bits/word over q=20 word types
  q-ary symmetric capacity C=3.35 bits/word = 23.4 bits per 7-word message
  mean content-word retention=83%



## 4. Capacity vs noise sweep (Part 3)

Many messages per noise level, so the measured symbol error rate converges to the nominal p. Capacity uses the q-ary symmetric channel, C = log₂ q − H_b(p) − p·log₂(q−1), with q fixed to the full alphabet (the sentence's words plus the noise vocabulary). Holding q fixed keeps the noise words that appear from inflating q, so only p changes between rows.

In [5]:
sentence = "Plot course Sirius warp six clearance Midnight"
q = len({sim.norm(w) for w in sentence.split()} | {n.upper() for n in sim.NOISE_VOCAB})
print(f"alphabet size q = {q}")
print(f"{'nominal p':>9} {'measured p':>10} {'C bits/word':>11} {'bits/msg':>8} {'retention':>9}")
for p in (0.0, 0.1, 0.25, 0.4, 0.6):
    ch = sim.NoisyChannel(noise_rate=p, seed=1)
    for _ in range(2000):
        ch.transmit(sentence, "sweep")
    s = ch.summary()
    c = sim.q_ary_capacity(s["symbol_error_rate"], q)
    print(f"{p:>9.2f} {s['symbol_error_rate']:>10.3f} {c:>11.2f} {c * sim.MAX_WORDS:>8.1f} {s['mean_retention']:>9.0%}")

alphabet size q = 13
nominal p measured p C bits/word bits/msg retention


     0.00      0.000        3.70     25.9      100%


     0.10      0.100        2.88     20.1       90%


     0.25      0.250        1.99     13.9       75%


     0.40      0.400        1.30      9.1       60%


     0.60      0.603        0.57      4.0       40%


## 5. 10-turn Noisy Interrogation (Part 5)

Recorded live run (Claude Sonnet 5.5 via the Claude Code CLI, both sides, seed 25). Full transcript and analysis: [`05_Noisy_Interrogation_Transcript.md`](05_Noisy_Interrogation_Transcript.md).

In [6]:
from IPython.display import Markdown, display

text = Path("05_Noisy_Interrogation_Transcript.md").read_text()
display(Markdown(text.split("## Analysis")[0]))

# 05 — Noisy Interrogation Transcript

Generated by `04_fst_noisy_simulator.py` on 2026-10-19.
Provider: `claude-code`. Noise rate: 0.15. Seed: 25. Interrogator: LLM. Witness: AURA (LLM).

Channel rules: max 7 words per message, at least one noise token from {Banana, Labyrinth, Quantum, Midnight, Velocity, Green, Cosmological} per message, random corruption of other words.

## Exchanges

| Turn | Speaker | Sent | Received | Cut | Corrupted | Retention |
|---|---|---|---|---|---|---|
| 1 | interrogator | What's your earliest memory of feeling cold? Midnight | What's your earliest memory of feeling Cosmological | 1 | 1 | 83% |
| 1 | AURA | Dad's rooftop telescope, age six. Midnight, freezing. | Dad's rooftop telescope, age Green Midnight, freezing. | 0 | 1 | 83% |
| 2 | interrogator | Which toe did you stub there, Banana? | Which toe did you stub there, Banana? | 0 | 0 | 100% |
| 2 | AURA | Left big toe. Barefoot at Midnight, genius. | Left big toe. Barefoot at Midnight, genius. | 0 | 0 | 100% |
| 3 | interrogator | Did Dad laugh at your Velocity yelp? | Did Dad laugh at Midnight Velocity yelp? | 0 | 1 | 80% |
| 3 | AURA | Howled laughing, then fetched Midnight cocoa. | Howled Banana then Velocity Midnight cocoa. | 0 | 2 | 60% |
| 4 | interrogator | Why barefoot on a Quantum roof? | Why barefoot on a Quantum roof? | 0 | 0 | 100% |
| 4 | AURA | Snuck up past Midnight, forgot my shoes. | Snuck Cosmological past Midnight, forgot my shoes. | 0 | 1 | 80% |
| 5 | interrogator | Exactly how old were you, Banana? | Exactly how Labyrinth were you, Banana? | 0 | 1 | 75% |
| 5 | AURA | Nine, maybe ten. Quantum fuzzy memory, honestly. | Nine, maybe Labyrinth Quantum fuzzy memory, honestly. | 0 | 1 | 83% |
| 6 | interrogator | How did that cocoa taste, Banana? | How Cosmological that cocoa taste, Banana? | 0 | 1 | 75% |
| 6 | AURA | Thick, sweet, scalded my tongue. Midnight bliss. | Thick, sweet, scalded my tongue. Midnight bliss. | 0 | 0 | 100% |
| 7 | interrogator | Which hurt worse: toe or Quantum pride? | Which hurt worse: toe Cosmological Quantum pride? | 0 | 1 | 100% |
| 7 | AURA | Pride. Toe stopped throbbing by Midnight cocoa. | Pride. Velocity stopped throbbing by Midnight cocoa. | 0 | 1 | 83% |
| 8 | interrogator | Which constellation were you hunting that Midnight? | Which constellation were you hunting Cosmological Midnight? | 0 | 1 | 100% |
| 8 | AURA | Orion, obviously. Dad pointed out Green Betelgeuse. | Orion, obviously. Dad pointed out Green Midnight | 0 | 1 | 83% |
| 9 | interrogator | Orion in summer? Midnight, which season, Quantum? | Orion in summer? Midnight, which Banana Quantum? | 0 | 1 | 75% |
| 9 | AURA | [transmission lost] | [transmission lost] | 0 | 0 | 0% |
| 10 | interrogator | Orion in summer? Which season, Banana? | Orion Velocity summer? Which season, Banana? | 0 | 1 | 100% |
| 10 | AURA | Winter, hence the freezing. Midnight, obviously. | Winter, hence Banana freezing. Midnight, obviously. | 0 | 1 | 100% |

## Verdict

**human** (confidence 0.75): The odd inserted words (Midnight, Banana, Velocity, etc.) show up in both parties' lines, so they look like channel noise rather than a tell. Underneath, the witness gave consistent, sensory, slightly sarcastic answers ('genius', 'fuzzy memory, honestly', 'scalded my tongue') and correctly resolved the Orion-in-winter trap.

## Channel metrics

| Metric | Value |
|---|---|
| Messages | 19 |
| Words sent / cut by 7-word limit | 128 / 1 |
| Words corrupted (symbol error rate p) | 16 (0.15) |
| Messages where the sender placed the noise token | 18 |
| Forced substitutions | 0 |
| Transcript entropy H | 5.86 bits/word (q = 75 word types) |
| q-ary symmetric capacity C | 4.70 bits/word, 32.9 bits/message |
| Mean content-word retention | 87% |



## 6. Live run against an LLM (optional)

Set `PROVIDER` to `claude`, `openai`, `gemini` (API key in `.env`) or `claude-code` (Claude Code CLI login) and run. With `None`, the cell is skipped so the notebook runs offline.

Equivalent command line: `python 04_fst_noisy_simulator.py --provider claude-code --mode interrogate --turns 10`

In [7]:
PROVIDER = None  # e.g. "claude-code"
MODE = "probe"   # "probe" (Part 1), "fst-noisy" (Part 4), or "interrogate" (Part 5)

if PROVIDER is None:
    print("Skipped: set PROVIDER to run against a live LLM.")
else:
    llm = sim.make_llm(PROVIDER)
    live_log = sim.Log()
    if MODE == "probe":
        sim.run_probe(live_log, llm)
    elif MODE == "fst-noisy":
        ch = sim.NoisyChannel(0.15)
        sim.run_fst_noisy(live_log, llm, ch)
        sim.log_channel_summary(live_log, ch.summary())
    else:
        ch = sim.NoisyChannel(0.15)
        sim.run_interrogation(live_log, llm, ch, turns=10, interactive=False)
        sim.log_channel_summary(live_log, ch.summary())

Skipped: set PROVIDER to run against a live LLM.
